# 🇻🇳 English → Vietnamese Translation Dataset Builder for Qwen2.5

This notebook builds a **high-quality instruction-tuning dataset** for fine-tuning Qwen2.5 on English→Vietnamese translation tasks.

## Pipeline Overview
1. **Load & validate** CSV input
2. **Clean text** — normalize, strip noise
3. **Build bilingual glossary** — extract domain terminology
4. **Build contextual retrieval** — FAISS-based semantic context
5. **Generate prompt/completion pairs** — Qwen2.5 instruction format
6. **Export** train/validation JSONL

## Input Format
```
en,vi,domain
"Hello world","Xin chào thế giới",general
```
**Supported domains:** `medical`, `business`, `IT`, `general`

## 📦 Cell 1: Install Dependencies

In [ ]:
# Install all required packages
import subprocess, sys

packages = [
    'pandas',
    'numpy',
    'tqdm',
    'sentence-transformers',
    'faiss-cpu',
    'spacy',
    'simalign',
    'scikit-learn',
    'transformers',
    'torch',
]

for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True
    )

# Download spaCy English model
print('Downloading spaCy en_core_web_sm model...')
subprocess.run(
    [sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'],
    capture_output=True
)

# Try installing awesome-align (optional, may not always succeed)
try:
    print('Attempting to install awesome-align (optional)...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', 'awesome-align', '-q'],
        capture_output=True, timeout=60
    )
    print('awesome-align installed.' if result.returncode == 0 else 'awesome-align not available, will use SimAlign fallback.')
except Exception:
    print('awesome-align not available, will use SimAlign fallback.')

print('\n✅ All packages installed!')

Installing pandas...
Installing numpy...
Installing tqdm...
Installing sentence-transformers...
Installing faiss-cpu...
Installing spacy...
Installing simalign...
Installing scikit-learn...
Installing transformers...
Installing torch...
Attempting to install awesome-align (optional)...
awesome-align installed.

✅ All packages installed!


## 📚 Cell 2: Imports & Configuration

In [ ]:
import os
import re
import json
import unicodedata
import warnings
import logging
from pathlib import Path
from typing import Optional
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

warnings.filterwarnings('ignore')
logging.getLogger('sentence_transformers').setLevel(logging.ERROR)

# ── Configuration ────────────────────────────────────────────────────────────
CONFIG = {
    # I/O
    'input_csv'       : '/content/sampled_10k_balanced.csv',          # Path to your input CSV
    'output_dir'      : './output',              # Where to save train/valid JSONL
    'train_file'      : 'train.json',
    'valid_file'      : 'valid.json',

    # Reproducibility
    'random_seed'     : 42,

    # Dataset split
    'train_ratio'     : 0.90,

    # Glossary
    'max_glossary_terms'   : 10,       # Max entries per sample
    'max_phrase_words'     : 5,        # Max words per glossary phrase
    'min_align_prob'       : 0.45,     # Min SimAlign confidence threshold

    # Context retrieval
    'context_top_k'        : 2,        # Top-K similar sentences
    'context_max_sentences': 2,        # Max context sentences in prompt

    # Models
    'embed_model' : 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',

    # Domains
    'valid_domains': {'medical', 'business', 'it', 'general'},
}

# Create output directory
Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)

np.random.seed(CONFIG['random_seed'])
print('✅ Configuration loaded.')
print(f"   Output directory : {CONFIG['output_dir']}")
print(f"   Embedding model  : {CONFIG['embed_model']}")

✅ Configuration loaded.
   Output directory : ./output
   Embedding model  : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 1️⃣ Cell 3: Load & Validate CSV

In [ ]:
def load_dataset(csv_path: str) -> pd.DataFrame:
    """
    Load the input CSV, validate structure, remove nulls/duplicates,
    normalize domains, and shuffle reproducibly.

    Expected columns: en, vi, domain
    """
    print(f'📂 Loading CSV: {csv_path}')

    # ── Read ─────────────────────────────────────────────────────────────────
    df = pd.read_csv(csv_path, encoding='utf-8', keep_default_na=True)
    print(f'   Raw rows        : {len(df):,}')

    # ── Validate columns ─────────────────────────────────────────────────────
    required = {'en', 'vi', 'domain'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    # ── Strip string whitespace ───────────────────────────────────────────────
    for col in ['en', 'vi', 'domain']:
        df[col] = df[col].astype(str).str.strip()

    # ── Remove null / empty rows ──────────────────────────────────────────────
    before = len(df)
    df.replace({'': np.nan, 'nan': np.nan, 'None': np.nan}, inplace=True)
    df.dropna(subset=['en', 'vi', 'domain'], inplace=True)
    print(f'   After null drop : {len(df):,}  (removed {before - len(df):,})')

    # ── Remove exact duplicates ───────────────────────────────────────────────
    before = len(df)
    df.drop_duplicates(subset=['en', 'vi'], inplace=True)
    print(f'   After dedup     : {len(df):,}  (removed {before - len(df):,})')

    # ── Normalize domain labels ───────────────────────────────────────────────
    df['domain'] = df['domain'].str.lower().str.strip()
    df['domain'] = df['domain'].apply(
        lambda d: d if d in CONFIG['valid_domains'] else 'general'
    )

    # ── Shuffle ───────────────────────────────────────────────────────────────
    df = df.sample(frac=1, random_state=CONFIG['random_seed']).reset_index(drop=True)
    print(f'   Final rows      : {len(df):,}')

    # ── Domain distribution ───────────────────────────────────────────────────
    print('\n📊 Domain distribution:')
    print(df['domain'].value_counts().to_string())

    return df


# ── Demo / Load ───────────────────────────────────────────────────────────────
# If the CSV doesn't exist yet, create a demo CSV so the notebook can run end-to-end
if not Path(CONFIG['input_csv']).exists():
    print('⚠️  Input CSV not found — generating demo data...')
    demo_rows = [
        {'en': 'The patient requires immediate surgery.', 'vi': 'Bệnh nhân cần phẫu thuật ngay lập tức.', 'domain': 'medical'},
        {'en': 'Please diagnose the symptoms carefully.', 'vi': 'Vui lòng chẩn đoán các triệu chứng một cách cẩn thận.', 'domain': 'medical'},
        {'en': 'The hospital has advanced MRI equipment.', 'vi': 'Bệnh viện có thiết bị MRI tiên tiến.', 'domain': 'medical'},
        {'en': 'Blood pressure should be monitored regularly.', 'vi': 'Huyết áp cần được theo dõi thường xuyên.', 'domain': 'medical'},
        {'en': 'The drug dosage must not exceed 500 mg daily.', 'vi': 'Liều thuốc không được vượt quá 500 mg mỗi ngày.', 'domain': 'medical'},
        {'en': 'Our quarterly revenue exceeded expectations.', 'vi': 'Doanh thu quý này của chúng tôi đã vượt kỳ vọng.', 'domain': 'business'},
        {'en': 'The merger was approved by the board.', 'vi': 'Việc sáp nhập đã được hội đồng quản trị phê duyệt.', 'domain': 'business'},
        {'en': 'Market analysis shows increasing demand.', 'vi': 'Phân tích thị trường cho thấy nhu cầu ngày càng tăng.', 'domain': 'business'},
        {'en': 'The contract terms are subject to negotiation.', 'vi': 'Các điều khoản hợp đồng có thể được thương lượng.', 'domain': 'business'},
        {'en': 'Profit margins improved in Q3.', 'vi': 'Biên lợi nhuận được cải thiện trong quý 3.', 'domain': 'business'},
        {'en': 'The operating system crashed due to a memory leak.', 'vi': 'Hệ điều hành bị sập do rò rỉ bộ nhớ.', 'domain': 'IT'},
        {'en': 'GPU utilization reached 98% during training.', 'vi': 'Mức sử dụng GPU đạt 98% trong quá trình huấn luyện.', 'domain': 'IT'},
        {'en': 'Deploy the container using Docker Compose.', 'vi': 'Triển khai container bằng Docker Compose.', 'domain': 'IT'},
        {'en': 'The API endpoint returns a JSON response.', 'vi': 'Endpoint API trả về phản hồi JSON.', 'domain': 'IT'},
        {'en': 'Machine learning models require large datasets.', 'vi': 'Các mô hình học máy cần tập dữ liệu lớn.', 'domain': 'IT'},
        {'en': 'The weather is beautiful today.', 'vi': 'Thời tiết hôm nay thật đẹp.', 'domain': 'general'},
        {'en': 'Could you please pass the salt?', 'vi': 'Bạn có thể chuyền muối giúp tôi không?', 'domain': 'general'},
        {'en': 'She reads books every evening before bed.', 'vi': 'Cô ấy đọc sách mỗi buổi tối trước khi ngủ.', 'domain': 'general'},
        {'en': 'The train departs at seven in the morning.', 'vi': 'Tàu khởi hành lúc bảy giờ sáng.', 'domain': 'general'},
        {'en': 'Learning a new language takes time and practice.', 'vi': 'Học một ngôn ngữ mới đòi hỏi thời gian và luyện tập.', 'domain': 'general'},
    ]
    pd.DataFrame(demo_rows).to_csv(CONFIG['input_csv'], index=False, encoding='utf-8')
    print(f'   Demo CSV created: {CONFIG["input_csv"]}  ({len(demo_rows)} rows)')

df = load_dataset(CONFIG['input_csv'])
df.head(3)

📂 Loading CSV: /content/sampled_10k_balanced.csv
   Raw rows        : 10,000
   After null drop : 10,000  (removed 0)
   After dedup     : 9,966  (removed 34)
   Final rows      : 9,966

📊 Domain distribution:
domain
general     2500
medical     2492
it          2488
business    2486


,en,vi,domain
0,"For the next two-and-a-half years, he earned a...","Trong 2 năm rưỡi, Vaughan kiếm được một hợp đồ...",general
1,Gustavo and Kaure moved quietly through the ho...,Gustavo và Kauren yên lặng tới lui trong căn n...,medical
2,"In the early 1970s, an industrious advertising...","Vào đầu những năm 1970, julie ray, người phụ t...",medical


## 2️⃣ Cell 4: Text Cleaning

In [ ]:
# Precompile regex patterns for performance
_ROMAN   = r'^(?:M{0,4}(?:CM|CD|D?C{0,3})(?:XC|XL|L?X{0,3})(?:IX|IV|V?I{0,3}))'
_BULLETS = re.compile(
    r'^(?:'
    r'\d{1,3}[.):]\s*'           # 1. 1) 1:
    r'|[a-zA-Z][.):]\s*'          # a. a) a:
    r'|\([a-zA-Z0-9]{1,3}\)\s*'  # (a) (1) (ii)
    r'|' + _ROMAN + r'[.):]\s*'   # Roman numerals
    r'|[-*•·▪▸►➤]\s+'            # Bullet symbols
    r')',
    re.IGNORECASE | re.MULTILINE
)

_UNICODE_QUOTES = str.maketrans({
    '\u2018': "'", '\u2019': "'",   # curly single quotes
    '\u201c': '"', '\u201d': '"',   # curly double quotes
    '\u201e': '"', '\u201f': '"',
    '\u2039': "'", '\u203a': "'",
    '\u00ab': '"', '\u00bb': '"',   # «»
})


def clean_text(text: str) -> str:
    """
    Robust text cleaning for both English and Vietnamese:
      - Remove numbering prefixes, roman numerals, bullet symbols
      - Normalize unicode quotes
      - Normalize whitespace
      - Preserve punctuation and sentence meaning
    """
    if not isinstance(text, str):
        return ''

    # ── NFC normalisation ─────────────────────────────────────────────────────
    text = unicodedata.normalize('NFC', text)

    # ── Normalize unicode quotes ──────────────────────────────────────────────
    text = text.translate(_UNICODE_QUOTES)

    # ── Remove bullet/numbering prefixes ─────────────────────────────────────
    # Apply on each line, then rejoin
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        # Remove the leading bullet/numbering pattern
        stripped = _BULLETS.sub('', stripped).strip()
        if stripped:
            cleaned_lines.append(stripped)

    text = ' '.join(cleaned_lines)

    # ── Collapse multiple spaces / tabs ───────────────────────────────────────
    text = re.sub(r'[ \t]+', ' ', text)

    # ── Remove zero-width / control chars (keep newline) ─────────────────────
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\u200b-\u200f\ufeff]', '', text)

    return text.strip()


# ── Apply cleaning ────────────────────────────────────────────────────────────
print('🧹 Cleaning text columns...')
df['en'] = df['en'].progress_apply(clean_text)
df['vi'] = df['vi'].progress_apply(clean_text)

# Drop any rows that became empty after cleaning
before = len(df)
df = df[(df['en'].str.strip() != '') & (df['vi'].str.strip() != '')].reset_index(drop=True)
print(f'   Rows after cleaning: {len(df):,}  (dropped {before - len(df):,} empty rows)')

# ── Sanity check ─────────────────────────────────────────────────────────────
print('\n🔎 Sample cleaned rows:')
for _, row in df.head(3).iterrows():
    print(f'  EN: {row["en"]}')
    print(f'  VI: {row["vi"]}')
    print(f'  Domain: {row["domain"]}\n')

🧹 Cleaning text columns...


  0%|          | 0/9966 [00:00<?, ?it/s]

  0%|          | 0/9966 [00:00<?, ?it/s]

   Rows after cleaning: 9,966  (dropped 0 empty rows)

🔎 Sample cleaned rows:
  EN: For the next two-and-a-half years, he earned a living performing weekly at a popular venue in town, the Soap Creek Saloon, and ultimately the newly opened Antone 's, widely known as Austin's " home of the blues ".
  VI: Trong 2 năm rưỡi, Vaughan kiếm được một hợp đồng biểu diễn hàng tuần tại một địa điểm nổi tiếng trong thành phố, Soap Creek Saloon và cuối cùng là Antone's mới khai trương, được biết đến rộng rãi như " Ngôi nhà Blues " của Austin..
  Domain: general

  EN: Gustavo and Kaure moved quietly through the house while I waited impatiently for them to finish and tried to pay attention to the happily-ever-after on the screen. I was starting to get sleepy) though, according to Edward, I'd slept half the day) when a rough voice startled me. Edward sat up, keeping me cradled against him, and answered Gustavo in flowing Portuguese. Gustavo nodded and walked quietly toward the front door. '
  VI: Gust

## 3️⃣ Cell 5: Load NLP Models (spaCy + SentenceTransformer)

In [ ]:
import spacy
from sentence_transformers import SentenceTransformer

print('🔄 Loading NLP models (this may take a minute on first run)...')

# spaCy English model
# NOTE: 'parser' must NOT be disabled — noun_chunks requires the dependency parse.
# We only disable components we genuinely don't need (senter, textcat).
nlp = spacy.load('en_core_web_sm', disable=['senter', 'textcat'])
print('   ✅ spaCy en_core_web_sm loaded (parser + NER enabled)')

# Multilingual sentence encoder
embed_model = SentenceTransformer(CONFIG['embed_model'])
print(f"   ✅ SentenceTransformer '{CONFIG['embed_model']}' loaded")

# Check for SimAlign
try:
    from simalign import SentenceAligner
    aligner = SentenceAligner(model='bert', token_type='bpe', matching_methods='m')
    ALIGNER_BACKEND = 'simalign'
    print('   ✅ SimAlign loaded')
except Exception as e:
    aligner = None
    ALIGNER_BACKEND = 'none'
    print(f'   ⚠️  SimAlign not available ({e}) — glossary will use NP-only extraction')

print(f'\n   Alignment backend: {ALIGNER_BACKEND}')

🔄 Loading NLP models (this may take a minute on first run)...
   ✅ spaCy en_core_web_sm loaded (parser + NER enabled)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ✅ SentenceTransformer 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' loaded


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-09 13:25:09,019 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


   ✅ SimAlign loaded

   Alignment backend: simalign


## 3️⃣ Cell 6: Glossary Builder

In [ ]:
# ── Vietnamese stopwords (common function words) ──────────────────────────────
VI_STOPWORDS = {
    'và', 'hoặc', 'nhưng', 'vì', 'nên', 'thì', 'là', 'có', 'không',
    'của', 'trong', 'trên', 'dưới', 'với', 'cho', 'đến', 'từ', 'về',
    'này', 'đó', 'kia', 'các', 'những', 'mọi', 'một', 'hai', 'ba',
    'được', 'đã', 'sẽ', 'đang', 'vẫn', 'cũng', 'đều', 'rất', 'quá',
    'mà', 'lại', 'ra', 'vào', 'lên', 'xuống', 'đi', 'đến', 'tới',
    'chỉ', 'mới', 'chưa', 'nếu', 'khi', 'sau', 'trước', 'giữa',
    'hơn', 'nhất', 'như', 'cùng', 'theo', 'qua', 'hay', 'hay là',
}

# ── Domain-specific stopwords that rarely need glossary entries ───────────────
EN_GENERIC_STOP = {
    'the', 'a', 'an', 'this', 'that', 'these', 'those',
    'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did',
    'will', 'would', 'could', 'should', 'may', 'might',
    'it', 'he', 'she', 'we', 'they', 'i', 'you',
    'in', 'on', 'at', 'to', 'for', 'of', 'by', 'with', 'from',
    'and', 'or', 'but', 'not', 'no', 'so', 'if', 'when',
}


def _extract_en_candidates(text: str, domain: str) -> list[str]:
    """
    Use spaCy to extract noun chunks, named entities, and abbreviations
    from the English sentence as glossary candidates.
    """
    doc = nlp(text)
    candidates = set()

    # Named entities
    for ent in doc.ents:
        phrase = ent.text.strip()
        words  = phrase.split()
        if 1 <= len(words) <= CONFIG['max_phrase_words'] and phrase.lower() not in EN_GENERIC_STOP:
            candidates.add(phrase)

    # Noun chunks (requires dependency parse — guarded for safety)
    if doc.has_annotation('DEP'):
        noun_chunk_iter = doc.noun_chunks
    else:
        noun_chunk_iter = []
    for chunk in noun_chunk_iter:
        phrase = chunk.text.strip()
        words  = phrase.split()
        if 1 <= len(words) <= CONFIG['max_phrase_words'] and phrase.lower() not in EN_GENERIC_STOP:
            candidates.add(phrase)

    # Abbreviations: uppercase tokens 2–8 chars (GPU, API, MRI, etc.)
    for token in doc:
        if token.is_upper and 2 <= len(token.text) <= 8 and token.is_alpha:
            candidates.add(token.text)

    # Domain-specific heuristic: technical compound nouns (IT)
    if domain == 'IT':
        tech_pattern = re.compile(
            r'\b(machine learning|deep learning|neural network|operating system|'
            r'source code|open source|cloud computing|data center|API|GPU|CPU|'
            r'database|server|container|microservice|endpoint|framework|runtime)\b',
            re.IGNORECASE
        )
        for m in tech_pattern.finditer(text):
            candidates.add(m.group(0))

    # Filter very short single words that are not abbreviations
    filtered = [
        c for c in candidates
        if not (len(c) <= 2 and not c.isupper())
    ]
    return filtered


def _align_with_simalign(en_text: str, vi_text: str) -> dict[str, str]:
    """
    Use SimAlign to produce word-level EN→VI alignments.
    Returns a dict mapping EN token index → VI token index.
    """
    if aligner is None:
        return {}
    try:
        en_tokens = en_text.split()
        vi_tokens = vi_text.split()
        if not en_tokens or not vi_tokens:
            return {}
        alignments = aligner.get_word_aligns(en_tokens, vi_tokens)
        # 'm' = match (highest recall argmax alignment)
        pairs = alignments.get('mwmf', alignments.get('inter', []))
        return {e: v for e, v in pairs}
    except Exception:
        return {}


def _find_vi_span(en_phrase: str, en_tokens: list[str],
                   vi_tokens: list[str], align_map: dict) -> Optional[str]:
    """
    Given an EN phrase and token-level alignment, find the corresponding
    Vietnamese span. Returns None if insufficient alignment coverage.
    """
    phrase_tokens = en_phrase.split()
    # Find the phrase's token positions in the EN sentence
    positions = []
    for i in range(len(en_tokens)):
        if en_tokens[i:i + len(phrase_tokens)] == phrase_tokens:
            positions = list(range(i, i + len(phrase_tokens)))
            break
    if not positions:
        return None

    # Map EN positions → VI positions
    vi_positions = sorted({align_map[p] for p in positions if p in align_map})
    if not vi_positions:
        return None

    # Require at least 50% of EN phrase tokens to be aligned
    coverage = len(vi_positions) / len(positions)
    if coverage < CONFIG['min_align_prob']:
        return None

    # Reconstruct the Vietnamese span (allow small gaps)
    if max(vi_positions) - min(vi_positions) > len(vi_positions) * 2:
        return None  # Too fragmented
    span_tokens = [vi_tokens[p] for p in range(min(vi_positions), max(vi_positions) + 1)]
    span = ' '.join(span_tokens)

    # Reject if span is a Vietnamese stopword
    if span.lower() in VI_STOPWORDS:
        return None
    if len(span.split()) > CONFIG['max_phrase_words']:
        return None

    return span


def build_glossary(en_text: str, vi_text: str, domain: str) -> dict[str, str]:
    """
    Build a bilingual glossary from one EN-VI sentence pair.

    Steps:
    1. Extract candidate EN phrases (NEs, noun chunks, abbreviations)
    2. Align EN→VI using SimAlign (or skip if unavailable)
    3. Filter low-confidence and noisy mappings
    4. Return up to max_glossary_terms entries

    Returns: dict { "en_phrase": "vi_phrase", ... }
    """
    glossary = {}

    en_candidates = _extract_en_candidates(en_text, domain)
    if not en_candidates:
        return glossary

    en_tokens = en_text.split()
    vi_tokens = vi_text.split()

    if ALIGNER_BACKEND == 'simalign':
        align_map = _align_with_simalign(en_text, vi_text)
    else:
        align_map = {}

    for phrase in en_candidates:
        if len(glossary) >= CONFIG['max_glossary_terms']:
            break

        # Skip generic stop tokens
        if phrase.lower() in EN_GENERIC_STOP:
            continue

        vi_span = None
        if align_map:
            vi_span = _find_vi_span(phrase, en_tokens, vi_tokens, align_map)

        # Fallback: for single-word abbreviations / proper nouns,
        # try a simple case-sensitive substring match in the VI text
        if vi_span is None and phrase.isupper() and len(phrase) <= 6:
            # Many technical abbreviations appear verbatim in VI text
            if phrase in vi_text:
                vi_span = phrase   # kept as-is in Vietnamese

        if vi_span and vi_span.strip():
            glossary[phrase] = vi_span.strip()

    return glossary


# ── Quick test ────────────────────────────────────────────────────────────────
sample_en = 'GPU utilization reached 98% during training.'
sample_vi = 'Mức sử dụng GPU đạt 98% trong quá trình huấn luyện.'
test_gloss = build_glossary(sample_en, sample_vi, 'IT')
print(f'🧪 Test glossary: {test_gloss}')

🧪 Test glossary: {'98%': '98%', 'GPU utilization': 'dụng GPU', 'GPU': 'GPU'}


## 3️⃣ Cell 7: Build Glossaries for All Rows

In [ ]:
print('📖 Building glossaries for all samples...')

glossaries = []
for _, row in tqdm(df.iterrows(), total=len(df), desc='Building glossary'):
    g = build_glossary(row['en'], row['vi'], row['domain'])
    glossaries.append(g)

df['glossary'] = glossaries

# Stats
non_empty = sum(1 for g in glossaries if g)
avg_terms = np.mean([len(g) for g in glossaries])
print(f'\n📊 Glossary stats:')
print(f'   Samples with ≥1 glossary entry : {non_empty:,} / {len(df):,} ({100*non_empty/len(df):.1f}%)')
print(f'   Average terms per sample        : {avg_terms:.2f}')
print(f'\n🔎 Sample glossary entry:')
for i, row in df[df['glossary'].apply(len) > 0].head(2).iterrows():
    print(f'  EN: {row["en"]}')
    print(f'  Glossary: {row["glossary"]}\n')

📖 Building glossaries for all samples...


Building glossary:   0%|          | 0/9966 [00:00<?, ?it/s]


📊 Glossary stats:
   Samples with ≥1 glossary entry : 9,763 / 9,966 (98.0%)
   Average terms per sample        : 5.80

🔎 Sample glossary entry:
  EN: For the next two-and-a-half years, he earned a living performing weekly at a popular venue in town, the Soap Creek Saloon, and ultimately the newly opened Antone 's, widely known as Austin's " home of the blues ".
  Glossary: {'Antone': "Antone's", 'the blues': 'nhà Blues', 'a popular venue': 'một địa điểm nổi', 'weekly': 'tuần'}

  EN: Gustavo and Kaure moved quietly through the house while I waited impatiently for them to finish and tried to pay attention to the happily-ever-after on the screen. I was starting to get sleepy) though, according to Edward, I'd slept half the day) when a rough voice startled me. Edward sat up, keeping me cradled against him, and answered Gustavo in flowing Portuguese. Gustavo nodded and walked quietly toward the front door. '
  Glossary: {'Edward': 'Edward', 'them': 'họ', 'Gustavo': 'Gustavo', 'the house':

## 4️⃣ Cell 8: Build Context Retrieval (FAISS)

In [ ]:
import faiss

def build_faiss_index(texts: list[str]) -> tuple:
    """
    Encode texts with the sentence transformer and build a FAISS flat L2 index.
    Returns (index, embeddings_array).
    """
    print(f'   Encoding {len(texts):,} sentences...')
    embeddings = embed_model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,   # cosine similarity via inner product
    )
    embeddings = np.array(embeddings, dtype='float32')
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)   # Inner Product → cosine (after L2-normalisation)
    index.add(embeddings)
    print(f'   FAISS index built: {index.ntotal} vectors, dim={dim}')
    return index, embeddings


# Build one FAISS index per domain for better contextual relevance
print('🔍 Building per-domain FAISS indices...')
domain_indices  = {}   # domain → faiss.Index
domain_texts    = {}   # domain → list[str]
domain_row_ids  = {}   # domain → list[original df index]

for domain in CONFIG['valid_domains']:
    mask = df['domain'] == domain
    if mask.sum() == 0:
        continue
    print(f'\n  Domain: {domain}  ({mask.sum()} sentences)')
    idx_list = df[mask].index.tolist()
    texts    = df.loc[mask, 'en'].tolist()
    faiss_idx, _ = build_faiss_index(texts)
    domain_indices[domain]  = faiss_idx
    domain_texts[domain]    = texts
    domain_row_ids[domain]  = idx_list

print('\n✅ All domain indices ready.')

🔍 Building per-domain FAISS indices...

  Domain: business  (2486 sentences)
   Encoding 2,486 sentences...


Batches:   0%|          | 0/39 [00:00<?, ?it/s]

   FAISS index built: 2486 vectors, dim=384

  Domain: medical  (2492 sentences)
   Encoding 2,492 sentences...


Batches:   0%|          | 0/39 [00:00<?, ?it/s]

   FAISS index built: 2492 vectors, dim=384

  Domain: it  (2488 sentences)
   Encoding 2,488 sentences...


Batches:   0%|          | 0/39 [00:00<?, ?it/s]

   FAISS index built: 2488 vectors, dim=384

  Domain: general  (2500 sentences)
   Encoding 2,500 sentences...


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

   FAISS index built: 2500 vectors, dim=384

✅ All domain indices ready.


## 4️⃣ Cell 9: Context Builder Function

In [ ]:
def build_context(df_index: int, current_text: str, domain: str) -> str:
    """
    Build a short context string (max 2 sentences) for a given sample.

    Strategy:
    1. Include the immediately preceding sentence in the same domain (if available)
    2. Retrieve top-K similar sentences from the domain FAISS index
    3. Deduplicate and trim

    Rules:
    - Never include the current sentence itself
    - Never include the Vietnamese translation (no target leakage)
    - Max CONFIG['context_max_sentences'] sentences
    """
    context_sentences = []
    seen = {current_text.strip().lower()}
    max_ctx = CONFIG['context_max_sentences']

    # ── 1. Previous sentence in same domain (sequential context) ─────────────
    if df_index > 0:
        prev_row = df.iloc[df_index - 1]
        if prev_row['domain'] == domain:
            prev_en = prev_row['en'].strip()
            key     = prev_en.lower()
            if key not in seen and len(prev_en) > 5:
                context_sentences.append(prev_en)
                seen.add(key)

    # ── 2. FAISS semantic retrieval ───────────────────────────────────────────
    if domain in domain_indices and len(context_sentences) < max_ctx:
        faiss_idx = domain_indices[domain]
        row_ids   = domain_row_ids[domain]
        query_vec = embed_model.encode([current_text], normalize_embeddings=True)
        query_vec = np.array(query_vec, dtype='float32')

        # Retrieve a few extra to account for deduplication
        k = min(CONFIG['context_top_k'] + 3, faiss_idx.ntotal)
        scores, indices = faiss_idx.search(query_vec, k)

        for score, local_idx in zip(scores[0], indices[0]):
            if len(context_sentences) >= max_ctx:
                break
            if local_idx < 0 or local_idx >= len(row_ids):
                continue
            orig_idx = row_ids[local_idx]
            if orig_idx == df_index:
                continue   # Skip the current sentence
            candidate = df.at[orig_idx, 'en'].strip()
            key = candidate.lower()
            if key in seen:
                continue
            if score < 0.3:   # Low relevance, skip
                continue
            context_sentences.append(candidate)
            seen.add(key)

    return '\n'.join(context_sentences[:max_ctx])


# ── Quick test ────────────────────────────────────────────────────────────────
test_idx  = 0
test_row  = df.iloc[test_idx]
test_ctx  = build_context(test_idx, test_row['en'], test_row['domain'])
print(f'🧪 Test context for row {test_idx}:')
print(f'  EN      : {test_row["en"]}')
print(f'  Context : {repr(test_ctx)}')

🧪 Test context for row 0:
  EN      : For the next two-and-a-half years, he earned a living performing weekly at a popular venue in town, the Soap Creek Saloon, and ultimately the newly opened Antone 's, widely known as Austin's " home of the blues ".
  Context : "The houses on Fell Street, Fredrica Bimmel's street, were termed waterfront on the weathered realtors' signs because their backyards ended at a slough, a backwater of the Licking River in Belvedere, Ohio, a Rust Belt town of 112,000, east of Columbus.\nTomorrow I'm going to his house in Playa del Carmen for the weekend."


## 5️⃣ Cell 10: Prompt/Completion Formatter

In [ ]:
def format_glossary(glossary: dict[str, str]) -> str:
    """
    Format glossary dict as a readable string for the prompt.
    Example: GPU: bộ xử lý đồ họa | operating system: hệ điều hành
    """
    if not glossary:
        return ''
    return ' | '.join(f'{en}: {vi}' for en, vi in glossary.items())


def format_sample(en_text: str, vi_text: str, domain: str,
                  glossary: dict, context: str) -> dict:
    """
    Build a single prompt/completion JSON object in Qwen2.5 instruction format.

    Output format:
    {
        "prompt": "Translate the following English text to Vietnamese.\n\nDomain: ...\n\n...",
        "completion": "Vietnamese translation only."
    }

    Rules:
    - Glossary section omitted if empty
    - Context section omitted if empty
    - Completion contains ONLY the Vietnamese text
    """
    parts = ['Translate the following English text to Vietnamese.']
    parts.append('')
    parts.append(f'Domain: {domain}')

    # Optional Glossary section
    gloss_str = format_glossary(glossary)
    if gloss_str:
        parts.append('')
        parts.append(f'Glossary: {gloss_str}')

    # Optional Context section
    ctx = context.strip()
    if ctx:
        parts.append('')
        parts.append('Context:')
        parts.append(ctx)

    # English source
    parts.append('')
    parts.append(f'English:')
    parts.append(en_text)

    # Vietnamese label
    parts.append('')
    parts.append('Vietnamese:')

    prompt = '\n'.join(parts)
    completion = vi_text.strip()

    return {'prompt': prompt, 'completion': completion}


# ── Verify format with a sample row ──────────────────────────────────────────
row    = df.iloc[0]
sample = format_sample(
    en_text    = row['en'],
    vi_text    = row['vi'],
    domain     = row['domain'],
    glossary   = row['glossary'],
    context    = build_context(0, row['en'], row['domain']),
)
print('📋 Sample formatted entry:')
print('──── PROMPT ────')
print(sample['prompt'])
print('──── COMPLETION ────')
print(sample['completion'])

📋 Sample formatted entry:
──── PROMPT ────
Translate the following English text to Vietnamese.

Domain: general

Glossary: Antone: Antone's | the blues: nhà Blues | a popular venue: một địa điểm nổi | weekly: tuần

Context:
The houses on Fell Street, Fredrica Bimmel's street, were termed waterfront on the weathered realtors' signs because their backyards ended at a slough, a backwater of the Licking River in Belvedere, Ohio, a Rust Belt town of 112,000, east of Columbus.
Tomorrow I'm going to his house in Playa del Carmen for the weekend.

English:
For the next two-and-a-half years, he earned a living performing weekly at a popular venue in town, the Soap Creek Saloon, and ultimately the newly opened Antone 's, widely known as Austin's " home of the blues ".

Vietnamese:
──── COMPLETION ────
Trong 2 năm rưỡi, Vaughan kiếm được một hợp đồng biểu diễn hàng tuần tại một địa điểm nổi tiếng trong thành phố, Soap Creek Saloon và cuối cùng là Antone's mới khai trương, được biết đến rộng rãi n

## 5️⃣ Cell 11: Generate All Prompt/Completion Pairs

In [ ]:
print('⚙️  Generating prompt/completion pairs for all samples...')

records = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc='Formatting samples'):
    context = build_context(idx, row['en'], row['domain'])
    record  = format_sample(
        en_text  = row['en'],
        vi_text  = row['vi'],
        domain   = row['domain'],
        glossary = row['glossary'],
        context  = context,
    )
    records.append(record)

print(f'✅ Generated {len(records):,} records.')

⚙️  Generating prompt/completion pairs for all samples...


Formatting samples:   0%|          | 0/9966 [00:00<?, ?it/s]

✅ Generated 9,966 records.


In [ ]:
print('⚙️  Generating prompt/completion pairs for all samples...')

records  = []
contexts = []   # collect to add back into df as a column

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Formatting samples'):
    context = build_context(idx, row['en'], row['domain'])
    contexts.append(context)
    record  = format_sample(
        en_text  = row['en'],
        vi_text  = row['vi'],
        domain   = row['domain'],
        glossary = row['glossary'],
        context  = context,
    )
    records.append(record)

# Store context back into the dataframe for CSV export
df['context'] = contexts

print(f'✅ Generated {len(records):,} records.')

⚙️  Generating prompt/completion pairs for all samples...


Formatting samples:   0%|          | 0/9966 [00:00<?, ?it/s]

✅ Generated 9,966 records.


## 6️⃣ Cell 12: Validate JSON Formatting

In [ ]:
def validate_record(record: dict, idx: int) -> list[str]:
    """
    Validate a single prompt/completion record.
    Returns a list of error strings (empty = valid).
    """
    errors = []

    if 'prompt' not in record:
        errors.append(f'[{idx}] Missing "prompt" key')
    if 'completion' not in record:
        errors.append(f'[{idx}] Missing "completion" key')

    if 'prompt' in record:
        if not isinstance(record['prompt'], str) or not record['prompt'].strip():
            errors.append(f'[{idx}] Empty or invalid prompt')
        if 'Vietnamese:' not in record['prompt']:
            errors.append(f'[{idx}] Prompt missing "Vietnamese:" label')
        if 'English:' not in record['prompt']:
            errors.append(f'[{idx}] Prompt missing "English:" label')

    if 'completion' in record:
        if not isinstance(record['completion'], str) or not record['completion'].strip():
            errors.append(f'[{idx}] Empty completion')

    # Roundtrip JSON serialization check
    try:
        roundtrip = json.loads(json.dumps(record, ensure_ascii=False))
        if roundtrip['prompt'] != record['prompt']:
            errors.append(f'[{idx}] JSON roundtrip mismatch in prompt')
    except Exception as e:
        errors.append(f'[{idx}] JSON serialization error: {e}')

    return errors


print('🔎 Validating all records...')
all_errors = []
for i, rec in enumerate(tqdm(records, desc='Validating')):
    all_errors.extend(validate_record(rec, i))

if all_errors:
    print(f'\n⚠️  Found {len(all_errors)} validation errors:')
    for e in all_errors[:10]:
        print(f'   {e}')
    if len(all_errors) > 10:
        print(f'   ... and {len(all_errors) - 10} more.')
else:
    print(f'✅ All {len(records):,} records are valid!')

🔎 Validating all records...


Validating:   0%|          | 0/9966 [00:00<?, ?it/s]

✅ All 9,966 records are valid!


## 6️⃣ Cell 13: Train/Validation Split & Export

In [ ]:
from sklearn.model_selection import train_test_split

def export_jsonl(data: list[dict], filepath: str) -> None:
    """
    Export a list of dicts to a JSONL file (one JSON object per line).
    UTF-8 encoded, ensure_ascii=False to preserve Vietnamese characters.
    """
    with open(filepath, 'w', encoding='utf-8') as f:
        for record in data:
            line = json.dumps(record, ensure_ascii=False)
            f.write(line + '\n')


# ── Split ─────────────────────────────────────────────────────────────────────
train_records, valid_records = train_test_split(
    records,
    test_size  = 1 - CONFIG['train_ratio'],
    random_state = CONFIG['random_seed'],
    shuffle    = True,
)

print(f'📐 Dataset split:')
print(f'   Train : {len(train_records):,} ({100*len(train_records)/len(records):.1f}%)')
print(f'   Valid : {len(valid_records):,} ({100*len(valid_records)/len(records):.1f}%)')

# ── Export ────────────────────────────────────────────────────────────────────
train_path = str(Path(CONFIG['output_dir']) / CONFIG['train_file'])
valid_path = str(Path(CONFIG['output_dir']) / CONFIG['valid_file'])

export_jsonl(train_records, train_path)
export_jsonl(valid_records, valid_path)

print(f'\n💾 Files saved:')
print(f'   {train_path}  ({Path(train_path).stat().st_size / 1024:.1f} KB)')
print(f'   {valid_path}  ({Path(valid_path).stat().st_size / 1024:.1f} KB)')

📐 Dataset split:
   Train : 8,969 (90.0%)
   Valid : 997 (10.0%)

💾 Files saved:
   output/train.json  (12645.5 KB)
   output/valid.json  (1414.6 KB)


##6️⃣ Cell 13b: Xuất CSV Enriched (en, vi, domain, glossary, context)

In [ ]:
def export_enriched_csv(df: pd.DataFrame, output_dir: str) -> str:
    """
    Xuất CSV gốc được bổ sung thêm 2 cột:
      - glossary : chuỗi JSON của dict thuật ngữ song ngữ
      - context  : chuỗi ngữ cảnh (các câu liên quan)

    Cột đầu ra: en | vi | domain | glossary | context
    Encoding: UTF-8 with BOM (utf-8-sig) để Excel hiển thị tiếng Việt đúng.
    """
    csv_path = str(Path(output_dir) / 'dataset_enriched.csv')

    # Chọn và sắp xếp cột
    export_cols = ['en', 'vi', 'domain', 'glossary', 'context']
    df_export = df[export_cols].copy()

    # Chuyển cột glossary (dict) → chuỗi JSON để lưu vào CSV
    df_export['glossary'] = df_export['glossary'].apply(
        lambda g: json.dumps(g, ensure_ascii=False) if isinstance(g, dict) else '{}'
    )

    # Đảm bảo context là chuỗi, thay NaN bằng chuỗi rỗng
    df_export['context'] = df_export['context'].fillna('').astype(str)

    # Xuất CSV — utf-8-sig để Excel mở đúng tiếng Việt
    df_export.to_csv(csv_path, index=False, encoding='utf-8-sig')

    return csv_path


csv_out = export_enriched_csv(df, CONFIG['output_dir'])
csv_size_kb = Path(csv_out).stat().st_size / 1024

print(f'📄 Enriched CSV saved:')
print(f'   Path : {csv_out}')
print(f'   Size : {csv_size_kb:.1f} KB')
print(f'   Rows : {len(df):,}')
print(f'   Cols : en | vi | domain | glossary | context')
print()

# ── Xem trước 3 dòng đầu ─────────────────────────────────────────────────────
import ast
preview = pd.read_csv(csv_out, encoding='utf-8-sig')
print('🔎 Preview (3 dòng đầu):')
for i, row in preview.head(3).iterrows():
    gloss_dict = json.loads(row['glossary']) if row['glossary'] else {}
    ctx_preview = (row['context'][:80] + '...') if len(str(row['context'])) > 80 else row['context']
    print(f'\n  [{i}] Domain  : {row["domain"]}')
    print(f'       EN      : {row["en"]}')
    print(f'       VI      : {row["vi"]}')
    print(f'       Glossary: {gloss_dict}')
    print(f'       Context : {ctx_preview}')

📄 Enriched CSV saved:
   Path : output/dataset_enriched.csv
   Size : 13107.4 KB
   Rows : 9,966
   Cols : en | vi | domain | glossary | context

🔎 Preview (3 dòng đầu):

  [0] Domain  : general
       EN      : For the next two-and-a-half years, he earned a living performing weekly at a popular venue in town, the Soap Creek Saloon, and ultimately the newly opened Antone 's, widely known as Austin's " home of the blues ".
       VI      : Trong 2 năm rưỡi, Vaughan kiếm được một hợp đồng biểu diễn hàng tuần tại một địa điểm nổi tiếng trong thành phố, Soap Creek Saloon và cuối cùng là Antone's mới khai trương, được biết đến rộng rãi như " Ngôi nhà Blues " của Austin..
       Glossary: {'Antone': "Antone's", 'the blues': 'nhà Blues', 'a popular venue': 'một địa điểm nổi', 'weekly': 'tuần'}
       Context : The houses on Fell Street, Fredrica Bimmel's street, were termed waterfront on t...

  [1] Domain  : medical
       EN      : Gustavo and Kaure moved quietly through the house while I w

## 7️⃣ Cell 14: Dataset Statistics

In [ ]:
# ── Text length stats ─────────────────────────────────────────────────────────
df['en_len'] = df['en'].apply(lambda x: len(x.split()))
df['vi_len'] = df['vi'].apply(lambda x: len(x.split()))
df['gloss_count'] = df['glossary'].apply(len)

print('=' * 55)
print('  📊  DATASET STATISTICS')
print('=' * 55)
print(f'  Total samples          : {len(df):,}')
print(f'  Train samples          : {len(train_records):,}')
print(f'  Validation samples     : {len(valid_records):,}')
print()
print(f'  EN avg. token length   : {df["en_len"].mean():.1f}  (min {df["en_len"].min()}, max {df["en_len"].max()})')
print(f'  VI avg. token length   : {df["vi_len"].mean():.1f}  (min {df["vi_len"].min()}, max {df["vi_len"].max()})')
print()
print(f'  Samples with glossary  : {(df["gloss_count"] > 0).sum():,} ({100*(df["gloss_count"] > 0).mean():.1f}%)')
print(f'  Avg. glossary terms    : {df["gloss_count"].mean():.2f}')
print()
print('  Domain distribution:')
for domain, cnt in df['domain'].value_counts().items():
    pct = 100 * cnt / len(df)
    bar = '█' * int(pct / 5)
    print(f'    {domain:10s}: {cnt:5,}  ({pct:5.1f}%)  {bar}')
print('=' * 55)

  📊  DATASET STATISTICS
  Total samples          : 9,966
  Train samples          : 8,969
  Validation samples     : 997

  EN avg. token length   : 41.0  (min 2, max 128)
  VI avg. token length   : 54.9  (min 3, max 150)

  Samples with glossary  : 9,763 (98.0%)
  Avg. glossary terms    : 5.80

  Domain distribution:
    general   : 2,500  ( 25.1%)  █████
    medical   : 2,492  ( 25.0%)  █████
    it        : 2,488  ( 25.0%)  ████
    business  : 2,486  ( 24.9%)  ████


## 7️⃣ Cell 15: Sample Output Preview

In [ ]:
print('🔎 Sample exported records:\n')

for i, rec in enumerate(records[:3]):
    print(f'─── Record {i+1} ───')
    print(json.dumps(rec, ensure_ascii=False, indent=2))
    print()

🔎 Sample exported records:

─── Record 1 ───
{
  "prompt": "Translate the following English text to Vietnamese.\n\nDomain: general\n\nGlossary: Antone: Antone's | the blues: nhà Blues | a popular venue: một địa điểm nổi | weekly: tuần\n\nContext:\nThe houses on Fell Street, Fredrica Bimmel's street, were termed waterfront on the weathered realtors' signs because their backyards ended at a slough, a backwater of the Licking River in Belvedere, Ohio, a Rust Belt town of 112,000, east of Columbus.\nTomorrow I'm going to his house in Playa del Carmen for the weekend.\n\nEnglish:\nFor the next two-and-a-half years, he earned a living performing weekly at a popular venue in town, the Soap Creek Saloon, and ultimately the newly opened Antone 's, widely known as Austin's \" home of the blues \".\n\nVietnamese:",
  "completion": "Trong 2 năm rưỡi, Vaughan kiếm được một hợp đồng biểu diễn hàng tuần tại một địa điểm nổi tiếng trong thành phố, Soap Creek Saloon và cuối cùng là Antone's mới khai tr

## 7️⃣ Cell 16: Verify Exported Files

In [ ]:
def verify_jsonl(filepath: str, label: str) -> None:
    """
    Verify a JSONL file:
    - All lines parseable as JSON
    - All records have 'prompt' and 'completion' keys
    - Vietnamese characters correctly preserved
    """
    errors  = []
    count   = 0
    vi_char_ok = True

    with open(filepath, 'r', encoding='utf-8') as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            count += 1
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                errors.append(f'Line {lineno}: JSON parse error: {e}')
                continue
            if 'prompt' not in obj or 'completion' not in obj:
                errors.append(f'Line {lineno}: Missing keys')
            # Check Vietnamese characters aren't escaped as \uXXXX
            if '\\u' in line and 'ộ' not in line and 'ế' not in line:
                vi_char_ok = False

    status = '✅' if not errors else '❌'
    print(f'{status} {label}  ({count:,} records, {Path(filepath).stat().st_size/1024:.1f} KB)')
    print(f'   Unicode (UTF-8) Vietnamese chars OK: {vi_char_ok}')
    if errors:
        for e in errors[:5]:
            print(f'   ERROR: {e}')


print('🔍 Verifying exported JSONL files...')
verify_jsonl(train_path, 'train.json')
verify_jsonl(valid_path, 'valid.json')

🔍 Verifying exported JSONL files...
✅ train.json  (8,969 records, 12645.5 KB)
   Unicode (UTF-8) Vietnamese chars OK: True
✅ valid.json  (997 records, 1414.6 KB)
   Unicode (UTF-8) Vietnamese chars OK: True


## ✅ Cell 17: Summary

In [ ]:
print("""
╔═══════════════════════════════════════════════════════╗
║        ✅  Pipeline Completed Successfully             ║
╚═══════════════════════════════════════════════════════╝
""")
print(f"  Input CSV       : {CONFIG['input_csv']}")
print(f"  Total samples   : {len(df):,}")
print(f"  Train records   : {len(train_records):,}  →  {train_path}")
print(f"  Valid records   : {len(valid_records):,}  →  {valid_path}")
print(f"  Aligner backend : {ALIGNER_BACKEND}")
print(f"  Embed model     : {CONFIG['embed_model']}")
print()
print("  Next steps:")
print("  1. Upload train.json and valid.json to your training environment")
print("  2. Fine-tune Qwen2.5 with a framework such as LLaMA-Factory or Axolotl")
print("  3. Use the 'prompt' field as model input, 'completion' as the target")
print()
print("  Example Axolotl dataset config:")
print("    datasets:")
print("      - path: output/train.json")
print("        type: completion  # prompt + completion fields")
print("        field_prompt: prompt")
print("        field_completion: completion")


╔═══════════════════════════════════════════════════════╗
║        ✅  Pipeline Completed Successfully             ║
╚═══════════════════════════════════════════════════════╝

  Input CSV       : /content/sampled_10k_balanced.csv
  Total samples   : 9,966
  Train records   : 8,969  →  output/train.json
  Valid records   : 997  →  output/valid.json
  Aligner backend : simalign
  Embed model     : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

  Next steps:
  1. Upload train.json and valid.json to your training environment
  2. Fine-tune Qwen2.5 with a framework such as LLaMA-Factory or Axolotl
  3. Use the 'prompt' field as model input, 'completion' as the target

  Example Axolotl dataset config:
    datasets:
      - path: output/train.json
        type: completion  # prompt + completion fields
        field_prompt: prompt
        field_completion: completion


In [ ]:
from google.colab import files

files.download('/content/output/train.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download('/content/output/valid.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>